In [ ]:
from physics.simulation import mcfm, msq
from physics.hzz import zpair, zz4l
from physics.hstar import c6

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,  # Don't override with default matplotlib fonts
    "pgf.preamble": "", 
    #     "\\\\usepackage{fontspec}"
    #     "\\\\setmainfont{Computer Modern Roman}"  # or Computer Modern if installed
})


In [2]:
COMPONENT = msq.Component.SIG

with open('data/xsec.json') as f:
    xsec = json.load(f)

xsec = {
    msq.Component.SBI : xsec['gg4l_sbi'],
    msq.Component.SIG : xsec['gg4l_sig'],
    msq.Component.INT : xsec['gg4l_int'],
    msq.Component.BKG : xsec['gg4l_bkg']
}

filenames = {
    msq.Component.SBI : 'data/ggZZ2e2m_sbi/events.csv',
    msq.Component.SIG : 'data/ggZZ2e2m_sig/events.csv',
    msq.Component.INT : 'data/ggZZ2e2m_int/events.csv',
    msq.Component.BKG : 'data/ggZZ2e2m_bkg/events.csv'
}

component_names = {
    msq.Component.SBI : 'SBI',
    msq.Component.SIG : 'SIG',
    msq.Component.INT : 'INT',
    msq.Component.BKG : 'BKG'
}

In [3]:
events = mcfm.from_csv(cross_section=xsec[COMPONENT], file_path=filenames[COMPONENT])
        
twoleptwonu = zz4l.TwoLepTwoNuSystem()

events = events.calculate(twoleptwonu)

In [23]:
xobs = events.kinematics['2l2v_mt'].to_numpy()
nxbins = 20
xmin, xmax = 100, 1100.0
# xbins = np.logspace(np.log10(xmin), np.log10(xmax), nxbins + 1)
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)

In [24]:
color_sm = 'black'
line_sm = '--'

In [25]:
c6_vals = [20, -20]
c6_colors = ['red', 'blue']
c6_lines = ['-', '-']

mod_c6 = c6.Modifier(baseline=COMPONENT, events=events, c6_values=[-10,-5,0,5,10])
wt_c6, prob_c6 = mod_c6.modify(c6_vals)

In [ ]:
h_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sm.fill(xobs, weight=events.weights)

h_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs, weight=wt_c6[:,i_c6])
    h_c6.append(h)

fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [2, 1]}, figsize=(6,6), sharex=True)

l_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    l_c6.append(ax1.stairs(h_c6[i_c6].values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6]))
l_sm = ax1.stairs(h_sm.values(), xbins, color=color_sm, linestyle=line_sm)

# ax1.set_yscale('log')

ax1.set_ylabel('$\\mathrm{d}\\sigma / \\mathrm{d} m_{\\mathrm{T}}^{ZZ}$ [pb]')

l_sm.set_label('$\\mathrm{SM}$')
for i_c6, c6_val in enumerate(c6_vals):
    sgn = '-' if c6_val < 0 else '+'
    l_c6[i_c6].set_label(f'$c_{{\\phi}} = {sgn}{abs(c6_val):d}$')
ax1.legend(frameon=False)

ax2.plot(xcenters, h_sm.values() / h_sm.values(), color=color_sm, linestyle=line_sm, drawstyle='steps-post')
for i_c6, c6_val in enumerate(c6_vals):
    ax2.plot(xcenters, h_c6[i_c6].values() / h_sm.values(), color=c6_colors[i_c6], linestyle=c6_lines[i_c6], drawstyle='steps-post')

ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$')
ax2.set_ylim(0.8,1.8)

ax2.set_xscale('linear')
ax1.set_xlabel('')
ax1.set_xlim(xmin, xmax)
ax2.set_xlim(xmin, xmax)
ax2.set_xlabel('$m_{4\\ell}$ [GeV]')

fig.tight_layout()
# ax1.tick_params(axis='x', labelbottom=False)

plt.savefig('mtzz.pdf')
plt.close()

/tmp/ipykernel_2977190/1589667346.py:27: RuntimeWarning: invalid value encountered in divide
  ax2.plot(xcenters, h_sm.values() / h_sm.values(), color=color_sm, linestyle=line_sm, drawstyle='steps-post')
/tmp/ipykernel_2977190/1589667346.py:29: RuntimeWarning: invalid value encountered in divide
  ax2.plot(xcenters, h_c6[i_c6].values() / h_sm.values(), color=c6_colors[i_c6], linestyle=c6_lines[i_c6], drawstyle='steps-post')
